# 05_multi_agent_patterns: Cooperative coder-reviewer feedback loops

This notebook implements a cooperative developer-reviewer multi-agent loop over coding challenges from Hugging Face's `openai/openai_humaneval` dataset, validating output iterations.

### Loop Design
1. **Developer Agent**: Completes the python function signature based on coding challenge prompts.
2. **Reviewer Agent**: Lints the syntax and reviews logic boundaries.
3. **Loop Verification**: Executes the dataset's official assertion check suites directly in-memory to confirm code correctness.

In [1]:
import os
from dotenv import load_dotenv
from datasets import load_dataset
from langchain_openai import ChatOpenAI

# Load keys from root .env
load_dotenv(dotenv_path=r"d:\\Study\\Prep\\.env")
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Load Humaneval dataset
try:
    dataset = load_dataset("openai/openai_humaneval", split="test")
    challenge = dataset[0]
    prompt_text = challenge["prompt"]
    test_code = challenge["test"]
    entry_point = challenge["entry_point"]
except Exception as e:
    print("Humaneval failed to load, using fallback challenge:", e)
    prompt_text = "def has_close_elements(numbers, threshold):\n"
    test_code = "def check(candidate):\n    assert candidate([1.0, 2.0, 3.0], 0.5) == False\n    print('Fallback check success!')"
    entry_point = "has_close_elements"

print("Programming Challenge Prompt:")
print(prompt_text)

D:\Study\Prep\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Programming Challenge Prompt:
from typing import List


def has_close_elements(numbers: List[float], threshold: float) -> bool:
    """ Check if in given list of numbers, are any two numbers closer to each other than
    given threshold.
    >>> has_close_elements([1.0, 2.0, 3.0], 0.5)
    False
    >>> has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)
    True
    """



In [2]:
# Run multi-agent iteration loop
print("Turn 1: Developer Agent writes code...")
developer_prompt = f"Complete the following python function. Return ONLY the executable python function definition code block, without any markdown wrappers or triple backticks:\n{prompt_text}"
developer_code = llm.invoke(developer_prompt).content

# Clean up code cell
developer_code = developer_code.replace("```python", "").replace("```", "").strip()
print("Developer Response:\n", developer_code)

print("\nTurn 1: Reviewer Agent lints and checks code...")
reviewer_prompt = f"Verify the following code for syntax and edge cases. Output a short critique and state 'APPROVED' if correct:\n{developer_code}"
reviewer_feedback = llm.invoke(reviewer_prompt).content
print("Reviewer Response:\n", reviewer_feedback)

# Execute test suite assertion
print("\nExecuting tests:")
local_scope = {}
exec(developer_code, globals(), local_scope)
candidate_fn = local_scope[entry_point]

# Define check suite
exec(test_code, globals(), local_scope)
local_scope["check"](candidate_fn)
print("\nSUCCESS: Candidate coding function passed all unit tests.")

Turn 1: Developer Agent writes code...


Developer Response:
 from typing import List

def has_close_elements(numbers: List[float], threshold: float) -> bool:
    """ Check if in given list of numbers, are any two numbers closer to each other than
    given threshold.
    >>> has_close_elements([1.0, 2.0, 3.0], 0.5)
    False
    >>> has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)
    True
    """
    for i in range(len(numbers)):
        for j in range(i + 1, len(numbers)):
            if abs(numbers[i] - numbers[j]) < threshold:
                return True
    return False

Turn 1: Reviewer Agent lints and checks code...


Reviewer Response:
 The provided code is syntactically correct and logically sound for the purpose it serves. Here’s a brief critique:

1. **Functionality**: The function `has_close_elements` correctly checks if any two numbers in the list are closer than the specified threshold. The nested loop effectively compares each pair of numbers.

2. **Efficiency**: The current implementation has a time complexity of O(n^2), which may not be efficient for large lists. If performance is a concern, consider using a more efficient algorithm, such as sorting the list first and then checking adjacent elements.

3. **Edge Cases**:
   - The function does not handle cases where the input list is empty. It should return `False` since there are no elements to compare.
   - It also does not handle cases where the threshold is negative. A negative threshold does not make sense in this context, and the function should ideally raise a ValueError in such cases.

4. **Docstring**: The docstring includes exampl

### Output Explanation & Verification

#### Executed Results Trace:
- **Developer Code Generation**: Generated the python function `has_close_elements(numbers, threshold)` that iterates through numeric lists to compare distance bounds.
- **Reviewer Inspection**: The reviewer validated the syntax and output format, confirming logic correctness.
- **Test Execution**: The testing assertions (`check(has_close_elements)`) executed without throwing any assertion errors, confirming code validity.

This demonstrates how multi-agent structures improve software generation reliability via loop-based peer review.